# Notebook 5: Activation Patching
**Kurs:** Mechanistic Interpretability  
**Modell:** EleutherAI/pythia-410m  
**Ziel:** Kausale Intervention — welche Schichten und Positionen tragen zur richtigen Antwort bei?

> **Aufgabe:** Implementiere die Zellen schrittweise. Dies ist das komplexeste Notebook\!
> 
> Hilfreiche Dokumentation:
> - PyTorch Hooks: https://pytorch.org/docs/stable/generated/torch.nn.Module.register_forward_hook.html
> - PyTorch Tensor clone: https://pytorch.org/docs/stable/generated/torch.Tensor.clone.html
> - Matplotlib Imshow: https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html

> **Empfehlung:** Lies zuerst den Hintergrund vollständig durch, bevor du beginnst.

## Hintergrund: Activation Patching — der Gold-Standard

### Das Problem mit Attention und Probing
- **Attention-Analyse**: Zeigt Korrelationen, keine Kausalität
- **Probing**: Zeigt, wo Information kodiert ist — nicht ob sie für die Ausgabe genutzt wird

### Die Idee
Activation Patching führt ein **kontrolliertes Experiment** durch:

```
Clean:     "The capital of France is"  →  P(" Paris") = hoch
Corrupted: "The capital of Xyzzy is"   →  P(" Paris") = niedrig
```

Dann: Korrupten Run wiederholen, aber die Aktivierung einer bestimmten **Schicht L** bei **Position P** mit der sauberen Version ersetzen.

**Wenn der Effekt stark ist** (P(" Paris") steigt nach dem Patchen):  
→ Diese Schicht/Position *enthält* die entscheidende Information

### Der Patching-Effekt
```
Effect(L, P) = logit_patched(" Paris") - logit_corrupt(" Paris")
```

Oder normalisiert:
```
Normalized_Effect(L, P) = (logit_patched - logit_corrupt) / (logit_clean - logit_corrupt)
```
Wert nahe 1.0 = vollständige Wiederherstellung der sauberen Vorhersage

## 1. Setup

In [ ]:
# TODO: Importiere torch, numpy, matplotlib
# TODO: Lade Tokenizer und Modell (wie in Notebook 1)
#
# MODEL_NAME = "EleutherAI/pythia-410m"
# n_layers = model.config.num_hidden_layers

## 2. Clean und Corrupted Prompts

In [ ]:
clean_prompt     = "The capital of France is"
corrupted_prompt = "The capital of Xyzzy is"
correct_answer   = " Paris"

# TODO: Bestimme die Token-ID des korrekten Antworttokens:
#   correct_tok_id = tokenizer.encode(correct_answer)[0]
#
# TODO: Tokenisiere BEIDE Prompts und stelle sicher, dass sie GLEICH LANG sind.
#   WICHTIG: Beide Inputs müssen dieselbe Sequenzlänge haben\!
#   Falls sie unterschiedlich lang sind, wähle einen anderen corrupted Prompt.
#
# TODO: Gib beide Token-Listen aus und vergleiche sie visuell.
# TODO: Führe Forward Passes für beide durch und vergleiche P(" Paris").
#
# Frage: Wie groß ist der Unterschied in P(" Paris") zwischen Clean und Corrupted?

## 3. Schritt 1: Clean-Aktivierungen cachen

In [ ]:
# TODO: Registriere Forward Hooks auf ALLEN Schichten des clean-Modells.
#   Ziel: Die Hidden States aller Schichten für den clean-Prompt speichern.
#
# Muster:
#   clean_cache = {}
#   for i, layer in enumerate(model.gpt_neox.layers):
#       def make_hook(layer_idx):
#           def hook_fn(module, input, output):
#               clean_cache[layer_idx] = output[0].detach().clone()
#               return output  # Wichtig: output unverändert zurückgeben\!
#           return hook_fn
#       layer.register_forward_hook(make_hook(i))
#
# TODO: Führe den Forward Pass durch.
# TODO: Entferne alle Hooks danach\!
#
# WICHTIG: .detach().clone() — warum beide?
#   .detach(): Löst vom Berechnungsgraphen (kein Gradient-Tracking)
#   .clone():  Erstellt eine echte Kopie (nicht nur eine Referenz)
#
# Dokumentation: https://pytorch.org/docs/stable/generated/torch.Tensor.clone.html

## 4. Schritt 2: Patching-Funktion implementieren

In [ ]:
# TODO: Implementiere patch_and_measure(patch_layer, patch_pos):
#
# Diese Funktion:
#   1. Registriert einen Hook auf model.gpt_neox.layers[patch_layer]
#   2. Der Hook ERSETZT die Aktivierung bei patch_pos mit clean_cache[patch_layer][0, patch_pos, :]
#   3. Führt den KORRUPTEN Forward Pass durch (corrupted_inputs\!)
#   4. Entfernt den Hook (try/finally für Sicherheit\!)
#   5. Gibt den Logit-Wert des correct_tok_id zurück
#
# Kritischer Code im Hook:
#   modified = output[0].clone()
#   modified[0, patch_pos, :] = clean_cache[target_layer][0, patch_pos, :]
#   return (modified,) + output[1:]   # Tuple-Struktur beibehalten\!
#
# WICHTIG: Pythia gibt (hidden_state, attention_probs, ...) als Tuple zurück.
#   Du MUSST das Tuple korrekt zurückgeben, sonst crasht das Modell.
#
# Tipp: Teste die Funktion erst mit einem einzelnen Wert, z.B.:
#   test_logit = patch_and_measure(patch_layer=10, patch_pos=-1)
#   print(f"Test Logit: {test_logit}")

## 5. Schritt 3: Alle Layer × Positionen patchen

In [ ]:
# TODO: Berechne den Patching-Effekt für ALLE Kombinationen:
#   results = np.zeros((n_layers, seq_len))
#   for layer in range(n_layers):
#       for pos in range(seq_len):
#           results[layer, pos] = patch_and_measure(layer, pos)
#
# WARNUNG: Dies dauert einige Minuten (24 Schichten × Sequenzlänge Forward Passes).
# Tipp: Gib alle 4 Schichten eine Fortschrittsmeldung aus.
#
# Nach der Schleife: Berechne den normalisierten Patching-Effekt:
#   normalized = (results - corr_base_logit) / (clean_logit - corr_base_logit)
#   0 = kein Effekt, 1 = vollständige Wiederherstellung

## 6. Heatmap visualisieren

In [ ]:
# TODO: Erstelle eine Heatmap der normalisierten Patching-Effekte:
#   - Y-Achse: Schichten (0 bis n_layers-1)
#   - X-Achse: Token-Positionen
#   - Farbe: normalisierter Effekt (cmap="RdBu", vmin=-0.2, vmax=1.0)
#   - X-Achsen-Labels: Token-Strings (clean_tokens)
#
# TODO: Markiere den stärksten Punkt (höchster normalisierter Effekt)
#   Tipp: np.unravel_index(np.argmax(normalized), normalized.shape)
#
# TODO: Erstelle daneben ein Balkendiagramm: maximaler Effekt pro Schicht
#   (max über alle Positionen für jede Schicht)
#
# Frage: Welche Schicht hat den größten Einfluss auf die Vorhersage?
# Frage: Welche Token-Position ist am wichtigsten?

## 7. Top-Patching-Punkte analysieren

In [ ]:
# TODO: Erstelle eine Rangliste der Top-10 (Layer, Position)-Kombinationen
#   nach normalisiertem Patching-Effekt.
#
# TODO: Beantworte: Befindet sich der wichtigste Patch-Punkt beim
#   Token "France"? Bei welcher Schicht? Was bedeutet das?
#
# Tipp: np.argsort(normalized.ravel())[::-1] sortiert absteigend
# Tipp: divmod(flat_idx, seq_len) gibt (layer, pos) zurück

## 8. BONUS: Head-Level Patching

In [ ]:
# BONUS-AUFGABE: Statt ganzer Schichten, patche einzelne Attention Heads.
#
# Für Head-Level Patching musst du in den Attention-Output eingreifen.
# In Pythia: model.gpt_neox.layers[i].attention
#
# Der Attention-Output hat die Form: (batch, seq, n_heads, head_dim)
# oder nach Reshape: (batch, seq, hidden_size)
#
# Herausforderung: Den Output eines einzelnen Heads zu isolieren und zu ersetzen.
#   head_dim = hidden_size // n_heads
#   start_idx = head * head_dim
#   end_idx   = (head + 1) * head_dim
#
# Welcher Attention Head ist am wichtigsten für die Vorhersage von " Paris"?
# Vergleiche dein Ergebnis mit der Attention-Analyse aus Notebook 3\!

## 9. Reflexionsfragen

In [ ]:
# Beantworte folgende Fragen:
#
# 1. Was ist der Unterschied zwischen Activation Patching und Attention-Analyse?
#    Welche Methode gibt kausalere Aussagen?
#
# 2. In welcher Schicht und an welcher Position ist der Patching-Effekt am stärksten?
#    Was schließt du darüber, wo das "France → Paris"-Wissen verarbeitet wird?
#
# 3. Vergleiche mit Notebook 4 (Probing):
#    Stimmt die Schicht mit dem höchsten Probing-Score überein mit dem höchsten
#    Patching-Effekt? Was bedeutet eine Diskrepanz (falls es eine gibt)?
#
# 4. Welche Limitierungen hat Activation Patching?
#    (Denke an: Rechenkosten, Abhängigkeit vom Prompt-Paar, Superposition)
#
# 5. Wie könnte man Activation Patching auf Attention-Head-Ebene erweitern?
#    Was würde man dadurch lernen?